# Bước 13 – Hướng 1: không có nhãn nào ở mạng mới

Lý thuyết từng bước, kèm ví dụ số, nằm trong tài liệu "TADP-IDS – Lý thuyết bước 13" Claude đã gửi.

So sánh trên cùng phần đánh giá, cùng cách đặt ngưỡng theo cửa sổ (bước 12):

| Nhóm | Phương pháp |
|---|---|
| Mốc đơn giản | trung bình đều 3 chuyên gia; chọn theo kiểm định nguồn; TADP (bài cũ) |
| Phương pháp đã công bố | CORAL (căn chỉnh tương quan); trọng số theo độ giống; tự huấn luyện bằng nhãn giả |
| Không cần mạng nguồn | Isolation Forest (rừng cô lập – phát hiện bất thường) |
| **Đề xuất** | nhãn giả chỉ lấy ở chỗ Isolation Forest và các chuyên gia **cùng chắc chắn** |
| So sánh với hướng 2 | M2 học từ 100 / 500 / 1.000 nhãn thật |

| Ô | Việc | Thời gian |
|---|---|---|
| 1 | Kiểm tra môi trường và mã nguồn | 1 phút |
| 2 | Chạy trên 4 mạng | 30–60 phút |

**Không cần GPU (card đồ họa).** Kết quả lưu vào `MyDrive/TADP_Q1/results/b13_khong_nhan/`. Nếu Colab ngắt, chạy lại Ô 2: mạng nào xong rồi sẽ được bỏ qua.

## Ô 1 – Kiểm tra môi trường và mã nguồn

In [1]:
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
CODE_DIR = "/content/drive/MyDrive/TADP_Q1/code"
sys.path.insert(0, CODE_DIR)
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
try:
    import lightgbm, pyarrow
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "pyarrow"], check=True)
EXPECTED = {"__init__.py": 125, "cic_data.py": 8579, "config.py": 3263, "data.py": 9956, "experts.py": 5934,
            "few_label.py": 15932, "label_free.py": 16327, "metrics.py": 4002, "preprocess.py": 3601,
            "shift.py": 3863, "window_threshold.py": 12240}
ok = True
for f, size in EXPECTED.items():
    p = os.path.join(CODE_DIR, "tadp", f)
    real = os.path.getsize(p) if os.path.exists(p) else None
    if real != size:
        ok = False
        print(f"   SAI: {f} – mong đợi {size} byte, thực tế {real}")
print("Mã nguồn:", "ĐẦY ĐỦ, đúng phiên bản bước 13" if ok else "CÓ LỖI – báo lại cho Claude")

Mounted at /content/drive
Mã nguồn: ĐẦY ĐỦ, đúng phiên bản bước 13


## Ô 2 – Chạy

In 4 dòng "chuyên gia … xong", rồi 4 dòng "[tên mạng] xong", rồi:
- Bảng 13.1: Macro-F1 trung bình 4 mạng, theo phương pháp và cách đặt ngưỡng (a = cố định, e = Otsu có chặn, h10 = chống sập 10%);
- Bảng 13.2: Macro-F1 theo từng mạng (ngưỡng h10);
- Bảng 13.3: AUROC – khả năng xếp hạng;
- Bảng 13.4: chất lượng nhãn giả (nhãn thật chỉ dùng để báo cáo);
- Bảng 13.5: các tín hiệu không cần nhãn có báo trước được thất bại không.

Chạy xong báo Claude.

In [2]:
import sys, warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.data import load_all, make_splits, common_features
from tadp.label_free import run_label_free
B = "/content/drive/MyDrive/TADP_Q1"
DOMAINS_CIC = {"CIC-IDS2017": ["2017"], "CSE-CIC-IDS2018": ["2018"], "CIC-ToN-IoT": ["ton"], "CIC-UNSW-NB15": ["unsw"]}
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b13_khong_nhan",
                 DOMAINS=DOMAINS_CIC, SPLIT_MODE="episode", LGBM_ROUNDS=200)
data = load_all(cfg)
splits, _ = make_splits(cfg, data)
cols = common_features(data, "no_id")
run_label_free(cfg, data, splits, cols, pv="P1_log")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CIC-IDS2017: 999,712 dòng, 87 cột, tỷ lệ tấn công = 0.2411
CSE-CIC-IDS2018: 999,829 dòng, 87 cột, tỷ lệ tấn công = 0.0559
CIC-ToN-IoT: 1,000,028 dòng, 82 cột, tỷ lệ tấn công = 0.5302
CIC-UNSW-NB15: 1,000,399 dòng, 82 cột, tỷ lệ tấn công = 0.0251
chuyên gia CIC-IDS2017     : xong (19 giây)
chuyên gia CSE-CIC-IDS2018 : xong (19 giây)
chuyên gia CIC-ToN-IoT     : xong (19 giây)
chuyên gia CIC-UNSW-NB15   : xong (14 giây)
[CIC-IDS2017] xong trong 265 giây
[CSE-CIC-IDS2018] xong trong 255 giây
[CIC-ToN-IoT] xong trong 256 giây
[CIC-UNSW-NB15] xong trong 243 giây
BẢNG 13.1 – Macro-F1 trung bình 4 mạng (a = cố định, e = Otsu có chặn, h10 = chống sập 10%)
 cach_nguong                   a_co_dinh  e_otsu_cua_so_co_chan  h_chong_sap_10
phuong_phap             k                                                     
CORAL                   0         0.471                 

cach_nguong                   a_co_dinh  e_otsu_cua_so_co_chan  h_chong_sap_10
phuong_phap             k                                                     
CORAL                   0         0.471                  0.365           0.365
DE_XUAT_dong_thuan      0         0.503                  0.504           0.504
M2_nhan_that            100       0.901                  0.890           0.896
                        500       0.827                  0.908           0.855
                        1000      0.784                  0.882           0.820
isolation_forest        0         0.466                  0.437           0.437
k0_TADP                 0         0.503                  0.446           0.446
k0_chon_kiem_dinh_nguon 0         0.492                  0.569           0.569
k0_trung_binh_deu       0         0.457                  0.387           0.387
nhan_gia_chi_IF         0         0.457                  0.456           0.456
trong_so_do_giong       0         0.433                  0.400           0.400
tu_huan_luyen           0         0.521                  0.522           0.522